In [ ]:
# Sample a series of random logical pauli gates

import stim
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, '.')
from StabilizerCode import StabilizerCode
from pauli_utils import pauli_str, LOGICAL_NAMES

# ---------------------------------------------------------------------------
# [[5,1,3]] perfect code
# ---------------------------------------------------------------------------
code_513 = StabilizerCode(
    n=5,
    generators=[
        (np.array([1,0,0,1,0]), np.array([0,1,1,0,0])),  # g1: XZZXI
        (np.array([0,1,0,0,1]), np.array([0,0,1,1,0])),  # g2: IXZZX
        (np.array([1,0,1,0,0]), np.array([0,0,0,1,1])),  # g3: XIXZZ
        (np.array([0,1,0,1,0]), np.array([1,0,0,0,1])),  # g4: ZXIXZ
    ],
    logical_x=(np.array([1,1,1,1,1]), np.array([0,0,0,0,0])),  # XXXXX
    logical_z=(np.array([0,0,0,0,0]), np.array([1,1,1,1,1])),  # ZZZZZ
    name='[[5,1,3]]',
)

# ---------------------------------------------------------------------------
# Simulation parameters
# ---------------------------------------------------------------------------
code = code_513
p_noise = 0.05          # single-qubit depolarizing rate
num_rounds = 50         # number of QEC rounds (random Pauli + noise + EC per round)
num_shots = 2000        # Monte Carlo shots
seed = 42

print(code)

# Stim MPP strings for stabilizer measurements
STAB_MPP = [
    'MPP X0*Z1*Z2*X3',   # g1 = XZZXI
    'MPP X1*Z2*Z3*X4',   # g2 = IXZZX
    'MPP X0*X2*Z3*Z4',   # g3 = XIXZZ
    'MPP Z0*X1*X3*Z4',   # g4 = ZXIXZ
]

def sample_random_logical_paulis(num_rounds, rng):
    """Sample a sequence of random logical Pauli gates (0=I, 1=X, 2=Y, 3=Z)."""
    return rng.integers(0, 4, size=num_rounds)

In [ ]:
# Encode the random logical pauli gates using a stabilizer code fault tolerantly
#
# Encoding uses stim.TableauSimulator:
#   1. Reset all qubits to |0⟩ (or |+⟩, |+i⟩ depending on which observable we track)
#   2. Project into code space via MPP of stabilizer generators
#   3. Apply correction to reach the +1 stabilizer sector
#   4. Fix logical eigenvalue if the correction flipped it

def apply_correction(sim, syndrome, code):
    """Apply the minimum-weight correction Pauli for a given syndrome."""
    cx, cz = code.corrections[syndrome]
    for q in range(code.n):
        if cx[q] and cz[q]:
            sim.y(q)
        elif cx[q]:
            sim.x(q)
        elif cz[q]:
            sim.z(q)

def measure_syndrome(sim):
    """Measure all stabilizer generators via MPP. Returns syndrome integer."""
    for sc in STAB_MPP:
        sim.do(stim.Circuit(sc))
    rec = sim.current_measurement_record()[-4:]
    return sum(int(b) << (3 - i) for i, b in enumerate(rec))

def prepare_logical_state(sim, basis='Z'):
    """
    Prepare a logical eigenstate in the code space.
    
    basis='Z': prepare |0_L⟩  (Z_L = +1)
    basis='X': prepare |+_L⟩  (X_L = +1)
    basis='Y': prepare |+i_L⟩ (Y_L = +1)
    """
    sim.reset(*range(code.n))
    
    if basis == 'X':
        for q in range(code.n):
            sim.h(q)
    elif basis == 'Y':
        for q in range(code.n):
            sim.h(q)
            sim.s(q)
    
    # Project into code space
    syn = measure_syndrome(sim)
    apply_correction(sim, syn, code)
    
    # Fix logical eigenvalue if correction flipped it
    obs_map = {'Z': stim.PauliString('ZZZZZ'),
               'X': stim.PauliString('XXXXX'),
               'Y': stim.PauliString('YYYYY')}
    if sim.peek_observable_expectation(obs_map[basis]) == -1:
        # Apply a logical Pauli that anticommutes with the observable to flip it
        if basis == 'Z':
            for q in range(code.n): sim.x(q)   # X_L flips Z_L
        elif basis == 'X':
            for q in range(code.n): sim.z(q)   # Z_L flips X_L
        elif basis == 'Y':
            for q in range(code.n): sim.x(q)   # X_L flips Y_L

def apply_transversal_pauli(sim, pauli_idx):
    """
    Apply a transversal logical Pauli gate.
    
    For [[5,1,3]]: X_L=XXXXX, Z_L=ZZZZZ, Y_L=YYYYY
    pauli_idx: 0=I, 1=X, 2=Y, 3=Z
    """
    if pauli_idx == 0:
        return
    gate = {1: 'x', 2: 'y', 3: 'z'}[pauli_idx]
    for q in range(code.n):
        getattr(sim, gate)(q)

print("Encoding functions defined.")

In [ ]:
# Sample a fault path on the physical qubit level using independent depolarising noise for single qubits
#
# Each round: random logical Pauli → depolarizing noise → syndrome measurement → correction
# The noise applies independent single-qubit depolarizing channel on each data qubit:
#   with prob p/3 each: X, Y, or Z error

def apply_depolarizing_noise(sim, p, rng):
    """Apply independent single-qubit depolarizing noise on all data qubits."""
    for q in range(code.n):
        r = rng.random()
        if r < p / 3:
            sim.x(q)
        elif r < 2 * p / 3:
            sim.y(q)
        elif r < p:
            sim.z(q)

def simulate_one_shot(num_rounds, p, pauli_sequence, basis, rng):
    """
    Run one Monte Carlo shot of the QEC memory experiment.
    
    At each round:
      1. Apply the scheduled transversal logical Pauli
      2. Apply depolarizing noise on all data qubits
      3. Measure syndromes and apply correction
    
    Returns: array of shape (num_rounds+1,) with the logical observable
             eigenvalue at each step (corrected for intended Pauli flips).
    """
    sim = stim.TableauSimulator()
    prepare_logical_state(sim, basis=basis)
    
    obs_str = {'Z': 'ZZZZZ', 'X': 'XXXXX', 'Y': 'YYYYY'}[basis]
    obs = stim.PauliString(obs_str)
    
    # Track the intended sign of the observable after applying logical Paulis
    # X_L anticommutes with Z_L and Y_L; Z_L anticommutes with X_L and Y_L
    # Pauli P flips observable O iff P anticommutes with O
    anticommutes = {
        'Z': {1, 2},   # X_L and Y_L flip Z_L
        'X': {2, 3},   # Y_L and Z_L flip X_L
        'Y': {1, 3},   # X_L and Z_L flip Y_L
    }
    flippers = anticommutes[basis]
    
    intended_sign = 1
    results = np.zeros(num_rounds + 1)
    results[0] = sim.peek_observable_expectation(obs) * intended_sign
    
    for t in range(num_rounds):
        # 1. Apply random logical Pauli (transversal)
        p_idx = pauli_sequence[t]
        apply_transversal_pauli(sim, p_idx)
        if p_idx in flippers:
            intended_sign *= -1
        
        # 2. Depolarizing noise
        apply_depolarizing_noise(sim, p, rng)
        
        # 3. Syndrome measurement + correction
        syn = measure_syndrome(sim)
        apply_correction(sim, syn, code)
        
        # Record observable relative to intended sign
        actual = sim.peek_observable_expectation(obs)
        results[t + 1] = actual * intended_sign
    
    return results

print("Simulation functions defined.")

In [ ]:
# 1. Decode to see the logical errors
#
# Run Monte Carlo simulation for each logical observable (X, Y, Z).
# At each round, we record whether the logical state matches the intended state.
# ⟨O_L⟩ = +1 means no logical error, ⟨O_L⟩ → 0 means maximally mixed.

rng = np.random.default_rng(seed)

# Storage: expectation_values[basis][round]
expectation_values = {}

for basis in ['Z', 'X', 'Y']:
    print(f"Simulating ⟨{basis}_L⟩ ({num_shots} shots, {num_rounds} rounds)...")
    accumulated = np.zeros(num_rounds + 1)
    
    for shot in range(num_shots):
        pauli_seq = sample_random_logical_paulis(num_rounds, rng)
        result = simulate_one_shot(num_rounds, p_noise, pauli_seq, basis, rng)
        accumulated += result
    
    expectation_values[basis] = accumulated / num_shots
    print(f"  ⟨{basis}_L⟩ at round 0: {expectation_values[basis][0]:.4f}")
    print(f"  ⟨{basis}_L⟩ at round {num_rounds}: {expectation_values[basis][-1]:.4f}")

# Also compute the Bloch vector length: r(t) = sqrt(⟨X⟩² + ⟨Y⟩² + ⟨Z⟩²)
# r=1 for pure state, r=0 for maximally mixed
bloch_length = np.sqrt(
    expectation_values['X']**2 +
    expectation_values['Y']**2 +
    expectation_values['Z']**2
)
print(f"\nBloch vector length at round 0: {bloch_length[0]:.4f}")
print(f"Bloch vector length at round {num_rounds}: {bloch_length[-1]:.4f}")

In [ ]:
# 2. Measure using the faulty circuit to see if the state reaches maximally mixed state
#    by measuring expectation values of X, Y, and Z (logical)

rounds = np.arange(num_rounds + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: individual expectation values
ax = axes[0]
for basis, color in [('Z', 'blue'), ('X', 'red'), ('Y', 'green')]:
    ax.plot(rounds, expectation_values[basis], '-', color=color, label=f'$\\langle {basis}_L \\rangle$')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='maximally mixed')
ax.set_xlabel('QEC round')
ax.set_ylabel('Expectation value')
ax.set_title(f'Logical qubit decoherence under depolarizing noise (p={p_noise})')
ax.legend()
ax.set_ylim(-0.2, 1.1)

# Right: Bloch vector length (purity measure)
ax = axes[1]
ax.plot(rounds, bloch_length, 'k-', linewidth=2, label='$|\\vec{r}|$')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('QEC round')
ax.set_ylabel('Bloch vector length')
ax.set_title(f'Approach to maximally mixed state ({code.name}, p={p_noise})')

# Fit exponential decay: r(t) ~ exp(-t/tau)
from scipy.optimize import curve_fit
def exp_decay(t, tau):
    return np.exp(-t / tau)
try:
    popt, _ = curve_fit(exp_decay, rounds[1:], bloch_length[1:], p0=[10])
    tau = popt[0]
    ax.plot(rounds, exp_decay(rounds, tau), 'r--', label=f'fit: $e^{{-t/\\tau}}$, $\\tau$={tau:.1f}')
    ax.legend()
    print(f"Decay time constant: tau = {tau:.2f} rounds")
except Exception as e:
    print(f"Fit failed: {e}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compare different noise rates
p_values = [0.01, 0.05, 0.1, 0.2]
num_rounds_compare = 50
num_shots_compare = 1000

fig, ax = plt.subplots(figsize=(8, 5))

for p_val in p_values:
    rng_cmp = np.random.default_rng(seed)
    acc = np.zeros(num_rounds_compare + 1)
    for shot in range(num_shots_compare):
        pauli_seq = sample_random_logical_paulis(num_rounds_compare, rng_cmp)
        result = simulate_one_shot(num_rounds_compare, p_val, pauli_seq, 'Z', rng_cmp)
        acc += result
    acc /= num_shots_compare
    ax.plot(np.arange(num_rounds_compare + 1), acc, label=f'p = {p_val}')

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('QEC round')
ax.set_ylabel('$\\langle Z_L \\rangle$')
ax.set_title(f'Decoherence rate vs noise strength ({code.name})')
ax.legend()
plt.tight_layout()
plt.show()